Knowledge Graph — Notebook 4
Building a Knowledge Graph from CSV Data:
 flow
 Real-world travel data
        ↓
CSV file
        ↓
Read CSV
        ↓
Understand rows
        ↓
Validate data
        ↓
Convert rows → triples
        ↓
Build Knowledge Graph
        ↓
Query
        ↓
Multi-hop traversal
        ↓
Reasoning

# CELL 01

# Knowledge Graph — Notebook 4
## Building a Travel Planning Knowledge Graph from CSV Data

### Running Problem: Travel Planning

In Notebook 1, we understood why we need a Knowledge Graph.

In Notebook 2, we built a Travel Planning Knowledge Graph manually.

In Notebook 3, we learned how to:

- query the graph
- identify graph patterns
- traverse relationships
- perform multi-hop traversal
- reason over connected information

Now we face a practical problem.

In a real application, we normally do not type every triple manually.

Our travel information may come from:

- CSV files
- databases
- APIs
- documents
- websites
- other data sources

In this notebook, we will start with a simple CSV file.

Our goal is:

CSV data
→ Triples
→ Knowledge Graph
→ Query
→ Traversal
→ Reasoning

# CELL 02

# Part 1 — Why CSV?

Suppose a travel agency gives us a file containing travel information.

For example:

Subject,Relationship,Object

Chennai,LOCATED_IN,Tamil Nadu
Mahabalipuram,NEAR,Chennai
Mahabalipuram,HAS_CATEGORY,Heritage

Instead of manually writing Python tuples, we can read this file.

Therefore:

CSV
→ read the rows
→ interpret each row as a triple
→ build the Knowledge Graph

The important idea is:

One CSV row can represent one Knowledge Graph triple.

# CELL 03

## Our CSV Structure

We will use three columns:

    Subject
    Relationship
    Object

Each row represents:

    Subject → Relationship → Object

For example:

    Mahabalipuram → NEAR → Chennai

This is exactly the triple representation we learned
in the previous notebooks.

Therefore, CSV gives us a convenient way to store
our triples in a tabular format.

In [1]:
# CELL 04

import csv

print("csv module imported successfully.")

csv module imported successfully.


# CELL 05

## Creating a Sample Travel CSV File

For this notebook, we will create our own CSV file using Python.

This avoids depending on an external file.

Later, in a real project, the CSV file may come from:

- a travel company
- a government tourism dataset
- an open-data portal
- an existing database export

The structure will remain the same.

In [2]:
# CELL 06

csv_data = """Subject,Relationship,Object
Chennai,LOCATED_IN,Tamil Nadu
Mahabalipuram,NEAR,Chennai
Mahabalipuram,HAS_CATEGORY,Heritage
Marina Beach,LOCATED_IN,Chennai
Chennai,CONNECTED_TO,Bengaluru
Bengaluru,CONNECTED_TO,Mysuru
Hotel SeaView,LOCATED_IN,Mahabalipuram
Hotel SeaView,PRICE_PER_NIGHT,3500
Pondicherry,NEAR,Chennai
Pondicherry,HAS_CATEGORY,Heritage
Hotel Heritage,LOCATED_IN,Pondicherry
Hotel Heritage,PRICE_PER_NIGHT,3000
"""

print(csv_data)

Subject,Relationship,Object
Chennai,LOCATED_IN,Tamil Nadu
Mahabalipuram,NEAR,Chennai
Mahabalipuram,HAS_CATEGORY,Heritage
Marina Beach,LOCATED_IN,Chennai
Chennai,CONNECTED_TO,Bengaluru
Bengaluru,CONNECTED_TO,Mysuru
Hotel SeaView,LOCATED_IN,Mahabalipuram
Hotel SeaView,PRICE_PER_NIGHT,3500
Pondicherry,NEAR,Chennai
Pondicherry,HAS_CATEGORY,Heritage
Hotel Heritage,LOCATED_IN,Pondicherry
Hotel Heritage,PRICE_PER_NIGHT,3000



# CELL 07

## Saving the CSV File

The variable csv_data currently contains text.

We will save it as:

    travel_triples.csv

This gives us a real CSV file that Python can read.

In [3]:
# CELL 08

with open("travel_triples.csv", "w", newline="", encoding="utf-8") as file:
    file.write(csv_data)

print("travel_triples.csv created successfully.")

travel_triples.csv created successfully.


# CELL 09

## Inspect the CSV File

Before processing the data, let us look at the file.

A CSV file is simply structured text.

For our problem:

    Row
    ↓
    Subject, Relationship, Object

Therefore, each data row potentially represents
one Knowledge Graph triple.

In [4]:
# CELL 10

with open("travel_triples.csv", "r", encoding="utf-8") as file:
    for line in file:
        print(line.strip())

Subject,Relationship,Object
Chennai,LOCATED_IN,Tamil Nadu
Mahabalipuram,NEAR,Chennai
Mahabalipuram,HAS_CATEGORY,Heritage
Marina Beach,LOCATED_IN,Chennai
Chennai,CONNECTED_TO,Bengaluru
Bengaluru,CONNECTED_TO,Mysuru
Hotel SeaView,LOCATED_IN,Mahabalipuram
Hotel SeaView,PRICE_PER_NIGHT,3500
Pondicherry,NEAR,Chennai
Pondicherry,HAS_CATEGORY,Heritage
Hotel Heritage,LOCATED_IN,Pondicherry
Hotel Heritage,PRICE_PER_NIGHT,3000


# CELL 11

# Part 2 — Reading CSV Using Python

Python provides the csv module for reading CSV files.

The basic process is:

    Open file
        ↓
    Create CSV reader
        ↓
    Read rows
        ↓
    Process each row

We will first read the header.

In [5]:
# CELL 12

with open("travel_triples.csv", "r", encoding="utf-8") as file:
    
    reader = csv.reader(file)
    
    header = next(reader)
    
    print("Header:")
    print(header)

Header:
['Subject', 'Relationship', 'Object']


# CELL 13

## Understanding the Header

The header tells us what each column means.

Position 0:

    Subject

Position 1:

    Relationship

Position 2:

    Object

Therefore, a row such as:

    Mahabalipuram,NEAR,Chennai

can be interpreted as:

    row[0] → Subject
    row[1] → Relationship
    row[2] → Object

In [6]:
# CELL 14

with open("travel_triples.csv", "r", encoding="utf-8") as file:
    
    reader = csv.reader(file)
    header = next(reader)
    
    for row in reader:
        print(row)

['Chennai', 'LOCATED_IN', 'Tamil Nadu']
['Mahabalipuram', 'NEAR', 'Chennai']
['Mahabalipuram', 'HAS_CATEGORY', 'Heritage']
['Marina Beach', 'LOCATED_IN', 'Chennai']
['Chennai', 'CONNECTED_TO', 'Bengaluru']
['Bengaluru', 'CONNECTED_TO', 'Mysuru']
['Hotel SeaView', 'LOCATED_IN', 'Mahabalipuram']
['Hotel SeaView', 'PRICE_PER_NIGHT', '3500']
['Pondicherry', 'NEAR', 'Chennai']
['Pondicherry', 'HAS_CATEGORY', 'Heritage']
['Hotel Heritage', 'LOCATED_IN', 'Pondicherry']
['Hotel Heritage', 'PRICE_PER_NIGHT', '3000']


# CELL 15

## From CSV Row to Triple

Look at this row:

    ['Mahabalipuram', 'NEAR', 'Chennai']

We can convert it directly into:

    ('Mahabalipuram', 'NEAR', 'Chennai')

Therefore:

CSV row
    ↓
Python list
    ↓
Python tuple
    ↓
Knowledge Graph triple

In [7]:
# CELL 16

with open("travel_triples.csv", "r", encoding="utf-8") as file:
    
    reader = csv.reader(file)
    header = next(reader)
    
    first_row = next(reader)

triple = (
    first_row[0],
    first_row[1],
    first_row[2]
)

print("CSV row:")
print(first_row)

print("\nTriple:")
print(triple)

CSV row:
['Chennai', 'LOCATED_IN', 'Tamil Nadu']

Triple:
('Chennai', 'LOCATED_IN', 'Tamil Nadu')


# CELL 17

# Part 3 — Convert the Entire CSV into Triples

We now know how to convert one row.

Let us convert every row.

The transformation is:

    CSV
     ↓
    rows
     ↓
    triples

After this step, our CSV data will have exactly
the same representation that we used in Notebook 2.

In [8]:
# CELL 18

triples = []

with open("travel_triples.csv", "r", encoding="utf-8") as file:
    
    reader = csv.reader(file)
    
    header = next(reader)
    
    for row in reader:
        subject = row[0]
        relationship = row[1]
        object_ = row[2]
        
        triples.append(
            (subject, relationship, object_)
        )

print("Number of triples:", len(triples))

Number of triples: 12


In [9]:
# CELL 19

for triple in triples:
    print(triple)

('Chennai', 'LOCATED_IN', 'Tamil Nadu')
('Mahabalipuram', 'NEAR', 'Chennai')
('Mahabalipuram', 'HAS_CATEGORY', 'Heritage')
('Marina Beach', 'LOCATED_IN', 'Chennai')
('Chennai', 'CONNECTED_TO', 'Bengaluru')
('Bengaluru', 'CONNECTED_TO', 'Mysuru')
('Hotel SeaView', 'LOCATED_IN', 'Mahabalipuram')
('Hotel SeaView', 'PRICE_PER_NIGHT', '3500')
('Pondicherry', 'NEAR', 'Chennai')
('Pondicherry', 'HAS_CATEGORY', 'Heritage')
('Hotel Heritage', 'LOCATED_IN', 'Pondicherry')
('Hotel Heritage', 'PRICE_PER_NIGHT', '3000')


# CELL 20

## We Have Reconstructed Our Knowledge

Notice something important.

In Notebook 2, we wrote:

    triples = [
        (...),
        (...),
        (...)
    ]

In this notebook, we did not manually type the triples.

Instead:

    CSV file
        ↓
    Python csv reader
        ↓
    triples

The resulting Knowledge Graph representation
is the same.

This is an important separation:

    DATA SOURCE
         ↓
    REPRESENTATION
         ↓
    KNOWLEDGE GRAPH

# CELL 21

# Part 4 — A Small Data Quality Problem

Real-world data is rarely perfect.

A CSV file may contain:

- empty values
- incomplete rows
- extra columns
- duplicate rows
- incorrect relationships
- unexpected data types

Therefore, before building a Knowledge Graph,
we should validate the data.

Let us first check whether every row has
exactly three columns.

In [10]:
# CELL 22

with open("travel_triples.csv", "r", encoding="utf-8") as file:
    
    reader = csv.reader(file)
    header = next(reader)
    
    for row_number, row in enumerate(reader, start=2):
        
        if len(row) != 3:
            print(
                "Invalid row:",
                row_number,
                row
            )
        else:
            print(
                "Valid row:",
                row_number,
                row
            )

Valid row: 2 ['Chennai', 'LOCATED_IN', 'Tamil Nadu']
Valid row: 3 ['Mahabalipuram', 'NEAR', 'Chennai']
Valid row: 4 ['Mahabalipuram', 'HAS_CATEGORY', 'Heritage']
Valid row: 5 ['Marina Beach', 'LOCATED_IN', 'Chennai']
Valid row: 6 ['Chennai', 'CONNECTED_TO', 'Bengaluru']
Valid row: 7 ['Bengaluru', 'CONNECTED_TO', 'Mysuru']
Valid row: 8 ['Hotel SeaView', 'LOCATED_IN', 'Mahabalipuram']
Valid row: 9 ['Hotel SeaView', 'PRICE_PER_NIGHT', '3500']
Valid row: 10 ['Pondicherry', 'NEAR', 'Chennai']
Valid row: 11 ['Pondicherry', 'HAS_CATEGORY', 'Heritage']
Valid row: 12 ['Hotel Heritage', 'LOCATED_IN', 'Pondicherry']
Valid row: 13 ['Hotel Heritage', 'PRICE_PER_NIGHT', '3000']


# CELL 23

## Why Does Validation Matter?

A Knowledge Graph is only as useful as the knowledge
we put into it.

For example:

    Hotel SeaView,LOCATED_IN,Mahabalipuram

is meaningful.

But:

    Hotel SeaView,,Mahabalipuram

has no relationship.

Therefore:

    Raw Data
       ↓
    Validation
       ↓
    Knowledge Graph

Data quality is part of Knowledge Graph construction.

In [11]:
# CELL 24

valid_triples = []

with open("travel_triples.csv", "r", encoding="utf-8") as file:
    
    reader = csv.reader(file)
    header = next(reader)
    
    for row_number, row in enumerate(reader, start=2):
        
        if len(row) != 3:
            print("Skipping invalid row:", row_number)
            continue
        
        subject, relationship, object_ = row
        
        if not subject.strip():
            print("Missing subject in row:", row_number)
            continue
        
        if not relationship.strip():
            print("Missing relationship in row:", row_number)
            continue
        
        if not object_.strip():
            print("Missing object in row:", row_number)
            continue
        
        valid_triples.append(
            (
                subject.strip(),
                relationship.strip(),
                object_.strip()
            )
        )

print("Valid triples:", len(valid_triples))

Valid triples: 12


# CELL 25

# Part 5 — A Subtle Issue: Data Types

There is an important difference between:

    3500

and:

    "3500"

The first is a number.

The second is text.

When we read a CSV file, values are normally read as text.

Therefore:

    PRICE_PER_NIGHT

will initially contain:

    "3500"

rather than:

    3500

In [12]:
# CELL 26

for triple in valid_triples:
    
    if triple[1] == "PRICE_PER_NIGHT":
        print(triple, type(triple[2]))

('Hotel SeaView', 'PRICE_PER_NIGHT', '3500') <class 'str'>
('Hotel Heritage', 'PRICE_PER_NIGHT', '3000') <class 'str'>


# CELL 27

## Why Does This Matter?

Suppose we want to ask:

    Is the hotel price less than ₹4000?

If the price is stored as text:

    "3500"

we cannot safely perform numeric comparison.

We need:

    3500

as an integer.

Therefore, when importing data into a Knowledge Graph,
we may need to convert certain literal values
to appropriate data types.

In [13]:
# CELL 28

typed_triples = []

for subject, relationship, object_ in valid_triples:
    
    if relationship == "PRICE_PER_NIGHT":
        object_ = int(object_)
    
    typed_triples.append(
        (subject, relationship, object_)
    )

print("Typed triples created:", len(typed_triples))

Typed triples created: 12


In [14]:
# CELL 29

for triple in typed_triples:
    
    if triple[1] == "PRICE_PER_NIGHT":
        print(triple, type(triple[2]))

('Hotel SeaView', 'PRICE_PER_NIGHT', 3500) <class 'int'>
('Hotel Heritage', 'PRICE_PER_NIGHT', 3000) <class 'int'>


# CELL 30

# Part 6 — Our CSV Data is Now a Knowledge Graph Representation

We now have:

    typed_triples

Each element has:

    Subject
    Relationship
    Object

For example:

    ('Hotel SeaView',
     'PRICE_PER_NIGHT',
     3500)

This is exactly the triple representation
we used in previous notebooks.

The source was different.

Notebook 2:

    Manually created triples

Notebook 4:

    CSV → triples

But the Knowledge Graph representation is the same.

In [15]:
# CELL 31

triples = typed_triples

for subject, relationship, object_ in triples:
    print(
        subject,
        "→",
        relationship,
        "→",
        object_
    )

Chennai → LOCATED_IN → Tamil Nadu
Mahabalipuram → NEAR → Chennai
Mahabalipuram → HAS_CATEGORY → Heritage
Marina Beach → LOCATED_IN → Chennai
Chennai → CONNECTED_TO → Bengaluru
Bengaluru → CONNECTED_TO → Mysuru
Hotel SeaView → LOCATED_IN → Mahabalipuram
Hotel SeaView → PRICE_PER_NIGHT → 3500
Pondicherry → NEAR → Chennai
Pondicherry → HAS_CATEGORY → Heritage
Hotel Heritage → LOCATED_IN → Pondicherry
Hotel Heritage → PRICE_PER_NIGHT → 3000


# CELL 32

# Part 7 — Build an Adjacency Graph

We will now build an adjacency representation.

For every triple:

    Subject → Relationship → Object

we store the relationship and object under the subject.

For example:

    Chennai → CONNECTED_TO → Bengaluru

becomes:

    graph["Chennai"] = [
        ("CONNECTED_TO", "Bengaluru")
    ]

In [16]:
# CELL 33

graph = {}

for subject, relationship, object_ in triples:
    
    if subject not in graph:
        graph[subject] = []
    
    graph[subject].append(
        (relationship, object_)
    )

graph

{'Chennai': [('LOCATED_IN', 'Tamil Nadu'), ('CONNECTED_TO', 'Bengaluru')],
 'Mahabalipuram': [('NEAR', 'Chennai'), ('HAS_CATEGORY', 'Heritage')],
 'Marina Beach': [('LOCATED_IN', 'Chennai')],
 'Bengaluru': [('CONNECTED_TO', 'Mysuru')],
 'Hotel SeaView': [('LOCATED_IN', 'Mahabalipuram'), ('PRICE_PER_NIGHT', 3500)],
 'Pondicherry': [('NEAR', 'Chennai'), ('HAS_CATEGORY', 'Heritage')],
 'Hotel Heritage': [('LOCATED_IN', 'Pondicherry'), ('PRICE_PER_NIGHT', 3000)]}

# CELL 34

## Inspect the Graph

The dictionary is now our simple
in-memory Knowledge Graph.

For example:

    graph["Chennai"]

should contain relationships such as:

    LOCATED_IN → Tamil Nadu
    CONNECTED_TO → Bengaluru

This is the same basic representation
we used in Notebook 2.

The difference is that this time
the graph was constructed from CSV data.

In [17]:
# CELL 35

print("Knowledge connected to Chennai:")

for relationship, object_ in graph.get("Chennai", []):
    print(
        "Chennai",
        "→",
        relationship,
        "→",
        object_
    )

Knowledge connected to Chennai:
Chennai → LOCATED_IN → Tamil Nadu
Chennai → CONNECTED_TO → Bengaluru


# CELL 36

# Part 8 — Query the Imported Knowledge Graph

Now comes an important test.

Can we ask the same questions
that we asked in Notebook 3?

Question:

    Where is Chennai located?

We should be able to answer it.

Why?

Because the source of the knowledge
should not affect how we query it.

Whether the triples came from:

    Python
    CSV
    Database
    API

the graph representation can remain the same.

In [18]:
# CELL 37

for subject, relationship, object_ in triples:
    
    if (
        subject == "Chennai"
        and relationship == "LOCATED_IN"
    ):
        print("Chennai is located in:", object_)

Chennai is located in: Tamil Nadu


# CELL 38

# Part 9 — Reuse the Generic Query Function

The query function from Notebook 3 can be reused.

This demonstrates an important software idea:

Knowledge representation
and
query mechanism

can be separated.

The data came from CSV.

But the query function does not need to know
where the data originally came from.

In [19]:
# CELL 39

def query_graph(triples, subject=None, relationship=None, object_=None):
    
    results = []
    
    for s, r, o in triples:
        
        if subject is not None and s != subject:
            continue
        
        if relationship is not None and r != relationship:
            continue
        
        if object_ is not None and o != object_:
            continue
        
        results.append((s, r, o))
    
    return results

In [20]:
# CELL 40

result = query_graph(
    triples,
    relationship="NEAR",
    object_="Chennai"
)

for triple in result:
    print(triple)

('Mahabalipuram', 'NEAR', 'Chennai')
('Pondicherry', 'NEAR', 'Chennai')


# CELL 41

# Part 10 — Multi-Hop Query

Let us repeat the multi-hop idea from Notebook 3.

Question:

    Can a traveller travel from Chennai to Mysuru
    through connected cities?

Our graph contains:

    Chennai
       ↓ CONNECTED_TO
    Bengaluru
       ↓ CONNECTED_TO
    Mysuru

The important point is:

The graph was loaded from CSV.

But the reasoning process remains the same.

In [21]:
# CELL 42

connections = {}

for subject, relationship, object_ in triples:
    
    if relationship == "CONNECTED_TO":
        
        if subject not in connections:
            connections[subject] = []
        
        connections[subject].append(object_)

print(connections)

{'Chennai': ['Bengaluru'], 'Bengaluru': ['Mysuru']}


# CELL 43

## BFS Traversal

We will again use Breadth-First Search.

The purpose is not to learn BFS in depth here.

The purpose is to demonstrate that once our data
has been converted into a graph representation,
we can apply graph algorithms to it.

In [22]:
# CELL 44

from collections import deque

def bfs_path(graph, start, target):
    
    queue = deque()
    queue.append((start, [start]))
    
    visited = set()

    while queue:
        
        current, path = queue.popleft()
        
        if current == target:
            return path
        
        if current in visited:
            continue
        
        visited.add(current)
        
        for neighbor in graph.get(current, []):
            
            if neighbor not in visited:
                queue.append(
                    (neighbor, path + [neighbor])
                )
    
    return None

In [23]:
# CELL 45

path = bfs_path(
    connections,
    "Chennai",
    "Mysuru"
)

print(" → ".join(path))

Chennai → Bengaluru → Mysuru


# CELL 46

# Part 11 — Travel Recommendation from CSV Data

Now let us answer a more useful question.

Question:

    Find affordable hotels in heritage destinations
    near Chennai.

The reasoning chain is:

Hotel
  ↓ LOCATED_IN
Destination
  ↓ NEAR
Chennai

and:

Destination
  ↓ HAS_CATEGORY
Heritage

and:

Hotel
  ↓ PRICE_PER_NIGHT
Price

We therefore need to combine several pieces
of information from the graph.

In [24]:
# CELL 47

near_chennai = []

for subject, relationship, object_ in triples:
    
    if (
        relationship == "NEAR"
        and object_ == "Chennai"
    ):
        near_chennai.append(subject)

print("Places near Chennai:")
for place in near_chennai:
    print("-", place)

Places near Chennai:
- Mahabalipuram
- Pondicherry


In [25]:
# CELL 48

heritage_places = []

for subject, relationship, object_ in triples:
    
    if (
        relationship == "HAS_CATEGORY"
        and object_ == "Heritage"
        and subject in near_chennai
    ):
        heritage_places.append(subject)

print("Heritage places near Chennai:")

for place in heritage_places:
    print("-", place)

Heritage places near Chennai:
- Mahabalipuram
- Pondicherry


In [26]:
# CELL 49

hotels = []

for subject, relationship, object_ in triples:
    
    if (
        relationship == "LOCATED_IN"
        and object_ in heritage_places
    ):
        hotels.append(
            (subject, object_)
        )

print("Hotels:")

for hotel, destination in hotels:
    print(
        hotel,
        "→",
        destination
    )

Hotels:
Hotel SeaView → Mahabalipuram
Hotel Heritage → Pondicherry


In [27]:
# CELL 50

budget = 4000

affordable_hotels = []

for hotel, destination in hotels:
    
    for subject, relationship, object_ in triples:
        
        if (
            subject == hotel
            and relationship == "PRICE_PER_NIGHT"
            and object_ < budget
        ):
            affordable_hotels.append(
                (hotel, destination, object_)
            )

print("Affordable hotels:")

for hotel, destination, price in affordable_hotels:
    print(
        hotel,
        "|",
        destination,
        "| ₹",
        price
    )

Affordable hotels:
Hotel SeaView | Mahabalipuram | ₹ 3500
Hotel Heritage | Pondicherry | ₹ 3000


# CELL 51

# Part 12 — What Have We Achieved?

We started with:

    travel_triples.csv

We then performed:

    CSV
     ↓
    Read rows
     ↓
    Validate rows
     ↓
    Convert rows to triples
     ↓
    Handle data types
     ↓
    Build graph
     ↓
    Query graph
     ↓
    Traverse graph
     ↓
    Multi-hop reasoning
     ↓
    Travel recommendation

This is the basic pipeline for constructing
a Knowledge Graph from structured data.

# CELL 52

# Part 13 — An Important Conceptual Distinction

A CSV file is a table.

For example:

    Subject | Relationship | Object
    ----------------------------------
    Chennai | LOCATED_IN   | Tamil Nadu
    Chennai | CONNECTED_TO | Bengaluru

A Knowledge Graph represents:

    Chennai
       ↓
    LOCATED_IN
       ↓
    Tamil Nadu

and:

    Chennai
       ↓
    CONNECTED_TO
       ↓
    Bengaluru

The SAME information can therefore be represented
in different forms.

CSV is a storage format.

A Knowledge Graph is a representation of
entities and their relationships.

This distinction will become very important
when we compare:

    CSV / Relational Database
             vs
    Knowledge Graph
             vs
    Vector Store

# CELL 53

# Part 14 — What is an Entity and What is a Literal?

Consider:

    Hotel SeaView → PRICE_PER_NIGHT → 3500

Here:

    Hotel SeaView

is an entity.

But:

    3500

is a literal value.

Similarly:

    Chennai → LOCATED_IN → Tamil Nadu

may be interpreted as:

    Chennai

    Tamil Nadu

as entities representing places, depending on
how the Knowledge Graph schema is designed.

The important lesson is:

Not every object in a graph has to represent
a real-world entity.

Some objects are literal values:

    3500
    3000
    "Chennai"
    dates
    numbers
    descriptions

# CELL 54

# Part 15 — Duplicate Knowledge

Real datasets may contain duplicate rows.

For example:

    Chennai,CONNECTED_TO,Bengaluru

may appear twice.

A simple Python list will store both.

But for many Knowledge Graph applications,
we may want unique triples.

One simple approach is to use a set.

In [28]:
# CELL 55

unique_triples = set(triples)

print("Original triples:", len(triples))
print("Unique triples:", len(unique_triples))

Original triples: 12
Unique triples: 12


# CELL 56

## Why Does This Matter?

Knowledge Graph construction is not simply:

    "Read the file."

A real pipeline often involves:

    Extract
       ↓
    Clean
       ↓
    Validate
       ↓
    Normalize
       ↓
    Deduplicate
       ↓
    Represent
       ↓
    Store
       ↓
    Query

This is why Knowledge Graph construction
is a data engineering as well as a knowledge
representation problem.

# CELL 57

# Part 16 — Exercise: Add More Travel Data

Create additional CSV rows for:

- Hyderabad
- Charminar
- Coorg
- Mangalore

For example:

    Hyderabad,CONNECTED_TO,Bengaluru
    Hyderabad,HAS_CATEGORY,Heritage
    Charminar,LOCATED_IN,Hyderabad

Then reload the CSV and rebuild the triples.

Ask:

> Can the existing query functions still work?

The expected answer should be YES.

Why?

Because the query mechanism works on the
triple representation, not on a particular city.

# CELL 58

# Part 17 — Exercise: Design Your Own CSV

Create a new CSV file containing at least:

    10 destinations
    5 hotels
    3 cities
    3 categories

Use relationships such as:

    LOCATED_IN
    NEAR
    CONNECTED_TO
    HAS_CATEGORY
    PRICE_PER_NIGHT

Then:

1. Load the CSV.
2. Convert it into triples.
3. Build the graph.
4. Query the graph.
5. Find a multi-hop path.
6. Answer one travel recommendation question.

# CELL 59

# Part 18 — Think Before You Code

Consider the question:

> Find a hotel under ₹3000 in a heritage destination
> near Chennai.

Before writing code, identify the graph patterns.

Pattern 1:

    ? → NEAR → Chennai

Pattern 2:

    ? → HAS_CATEGORY → Heritage

Pattern 3:

    Hotel → LOCATED_IN → ?

Pattern 4:

    Hotel → PRICE_PER_NIGHT → ?

The reasoning chain becomes:

    Hotel
       ↓
    Destination
       ↓
    Chennai

while also checking:

    Destination → Heritage

and:

    Hotel → Price < ₹3000

This is the same graph reasoning we learned
in Notebook 3.

# CELL 60

# Part 19 — The Bigger Picture

We have now learned two ways of obtaining
the same Knowledge Graph representation.

### Notebook 2

    Human
      ↓
    Manually create triples
      ↓
    Knowledge Graph

### Notebook 4

    CSV
      ↓
    Python
      ↓
    Triples
      ↓
    Knowledge Graph

The next question is:

> How does this compare with storing the same
> information in a relational database?

For example, we could store:

    Hotels
    Destinations
    Cities
    Prices
    Connections

in SQL tables.

So we now have three important approaches:

    Relational Database
             ↓
    Knowledge Graph
             ↓
    Vector Store

Each has strengths and weaknesses.

We should not conclude that one is always better.

The important question is:

> Which representation is appropriate for
> the problem we are trying to solve?

# CELL 61

# Final Takeaway

Notebook 4 taught us how structured data can become
Knowledge Graph knowledge.

The complete journey is:

    CSV
     ↓
    Read
     ↓
    Validate
     ↓
    Clean
     ↓
    Convert to Triples
     ↓
    Build Graph
     ↓
    Query
     ↓
    Traverse
     ↓
    Multi-Hop Reasoning
     ↓
    Answer

The key idea is:

> A CSV is a data source.
> A triple is a knowledge representation.
> A Knowledge Graph connects those representations
> through relationships.

We are now ready to compare different ways of
representing the SAME travel knowledge.

In [ ]:
# CELL 62



## What we learned

✓ CSV as a source of structured knowledge

✓ CSV rows → triples

✓ Data validation

✓ Data type conversion

✓ Duplicate handling

✓ Building an in-memory Knowledge Graph

✓ Querying the imported graph

✓ Graph traversal

✓ Multi-hop reasoning

✓ Travel recommendation using graph relationships

## Next Notebook

# KG-05 — Knowledge Graph vs Relational Database vs Vector Store

We will use the SAME Travel Planning problem.

The same knowledge will be represented in:

    1. Relational Database
    2. Knowledge Graph
    3. Vector Store

Then we will ask:

> Which representation is better for which type
> of question?

The goal is not to declare a winner.

The goal is to understand:

# Representation should match the problem.

KG-01
Why KG?
   ↓
KG-02
Build KG manually
   ↓
KG-03
Query + Traversal + Multi-hop
   ↓
KG-04
CSV → KG
   ↓
KG-05
KG vs Relational DB vs Vector Store
   ↓
KG-06
Neo4j + Cypher
   ↓
KG-07
RDF + SPARQL
   ↓
KG-08
Travel Planning KG Project
KG-05 especially important, because it prevents students from developing the misconception that “Knowledge Graph is always superior to a relational database or vector database.” The objective should be to teach why a particular representation is appropriate for a particular question